In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp04/ex_5.py ──
"""
Shared infrastructure for MLFP04 Exercise 5 — Association Rules.

Contains: synthetic Singapore retail transaction generator, category map,
rule dataclass helpers, and small polars utilities used by every technique
file in ``modules/mlfp04/solutions/ex_5/``.

Technique-specific code (Apriori from scratch, FP-Growth wrapper, rule
evaluation, feature engineering for classification) does NOT belong here —
it lives in the per-technique files.
"""

import asyncio
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import polars as pl

from kailash_ml import ExperimentTracker


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp04_ex5_association_rules"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE RETAIL PRODUCT CATALOGUE
# ════════════════════════════════════════════════════════════════════════
# 25 products grouped to mirror a typical HDB neighbourhood mini-mart basket.

PRODUCTS: list[str] = [
    "bread",
    "butter",
    "milk",
    "eggs",
    "rice",
    "noodles",
    "soy_sauce",
    "cooking_oil",
    "chicken",
    "fish",
    "coffee",
    "tea",
    "sugar",
    "condensed_milk",
    "biscuits",
    "chips",
    "soft_drink",
    "beer",
    "wine",
    "tissue",
    "shampoo",
    "soap",
    "detergent",
    "toothpaste",
    "bananas",
]

CATEGORY_MAP: dict[str, str] = {
    "bread": "breakfast",
    "butter": "breakfast",
    "eggs": "breakfast",
    "milk": "dairy",
    "condensed_milk": "dairy",
    "coffee": "beverage",
    "tea": "beverage",
    "soft_drink": "beverage",
    "sugar": "pantry",
    "rice": "pantry",
    "cooking_oil": "pantry",
    "soy_sauce": "pantry",
    "noodles": "pantry",
    "chicken": "protein",
    "fish": "protein",
    "beer": "alcohol",
    "wine": "alcohol",
    "chips": "snack",
    "biscuits": "snack",
    "bananas": "fruit",
    "shampoo": "personal_care",
    "soap": "personal_care",
    "toothpaste": "personal_care",
    "tissue": "household",
    "detergent": "household",
}

# Co-purchase bundles (items, probability). Models real behaviour:
# kaya-toast breakfast, kopi-C, household replenishment, beer+chips, etc.
BUNDLES: list[tuple[list[str], float]] = [
    (["bread", "butter", "eggs"], 0.15),
    (["coffee", "condensed_milk", "sugar"], 0.12),
    (["rice", "chicken", "soy_sauce"], 0.10),
    (["noodles", "eggs", "soy_sauce"], 0.08),
    (["beer", "chips"], 0.09),
    (["milk", "biscuits"], 0.07),
    (["shampoo", "soap", "toothpaste"], 0.06),
    (["tea", "sugar", "biscuits"], 0.05),
    (["wine", "chips", "biscuits"], 0.04),
    (["cooking_oil", "rice", "fish"], 0.06),
    (["detergent", "tissue", "soap"], 0.05),
    (["bananas", "milk", "eggs"], 0.05),
]

N_TRANSACTIONS_DEFAULT: int = 2500


# ════════════════════════════════════════════════════════════════════════
# TRANSACTION GENERATOR
# ════════════════════════════════════════════════════════════════════════


def _draw_basket(rng: np.random.Generator, bundle_probs: Iterable[float]) -> set[str]:
    """Draw one basket: each bundle fires with its probability, each item in
    a firing bundle is kept with 0.85 probability, plus a Poisson(2) number
    of random impulse items."""
    basket: set[str] = set()
    for (bundle_items, _), prob in zip(BUNDLES, bundle_probs):
        if rng.random() < prob:
            for item in bundle_items:
                if rng.random() < 0.85:
                    basket.add(item)
    n_random = rng.poisson(2)
    if n_random > 0:
        random_items = rng.choice(PRODUCTS, size=int(min(n_random, 5)), replace=False)
        basket.update(random_items)
    return basket


def generate_transactions(
    n: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> list[set[str]]:
    """Generate ``n`` non-empty synthetic Singapore retail transactions.

    Each transaction is a set of product strings. Bundles fire with their
    listed probability; each item inside a firing bundle is kept with 0.85
    probability (random drop-out) so support is noisy. A Poisson number of
    random items is added on top to simulate impulse buys. Empty draws are
    re-drawn, so exactly ``n`` baskets are returned.
    """
    rng = np.random.default_rng(seed)
    base_probs = [prob for _, prob in BUNDLES]
    transactions: list[set[str]] = []
    while len(transactions) < n:
        basket = _draw_basket(rng, base_probs)
        if basket:
            transactions.append(basket)
    return transactions


# Habit model for two-trip shoppers (used by Exercise 5.4). A shopper is a
# "regular" for a bundle with probability HABIT_RATE_MULTIPLIER x the
# bundle's base rate; regulars fire that bundle on HABIT_FIRE_PROB of their
# trips, everyone else on CASUAL_FIRE_PROB of their trips.
HABIT_RATE_MULTIPLIER: float = 1.5
HABIT_FIRE_PROB: float = 0.6
CASUAL_FIRE_PROB: float = 0.03


def generate_shopper_trips(
    n_shoppers: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> tuple[list[set[str]], list[set[str]]]:
    """Two consecutive baskets per shopper: ``(this_trip, next_trip)``.

    Both trips are drawn independently from the SAME per-shopper habit
    profile, so this trip carries real but noisy evidence about the next
    one. The next trip is never used to build features — it only supplies
    a prediction target that is NOT a function of this trip's items.
    """
    rng = np.random.default_rng(seed)
    base = np.array([prob for _, prob in BUNDLES])
    this_trip: list[set[str]] = []
    next_trip: list[set[str]] = []
    while len(this_trip) < n_shoppers:
        habit = rng.random(len(BUNDLES)) < base * HABIT_RATE_MULTIPLIER
        probs = np.where(habit, HABIT_FIRE_PROB, CASUAL_FIRE_PROB)
        first = _draw_basket(rng, probs)
        second = _draw_basket(rng, probs)
        if first:
            this_trip.append(first)
            next_trip.append(second)
    return this_trip, next_trip


def transactions_to_onehot(transactions: list[set[str]]) -> pl.DataFrame:
    """One-row-per-transaction boolean matrix (columns = sorted PRODUCTS).

    Polars-native. mlxtend needs pandas, so callers convert with
    ``.to_pandas()`` at the mlxtend call site only.
    """
    all_items = sorted(PRODUCTS)
    rows = [{item: (item in txn) for item in all_items} for txn in transactions]
    return pl.DataFrame(rows)


def product_frequency(transactions: Iterable[set[str]]) -> dict[str, int]:
    """Count how many transactions contain each product."""
    counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            counts[item] += 1
    return dict(counts)


def print_transaction_summary(transactions: list[set[str]]) -> None:
    """One-liner summary + top 10 product frequency. Used by every file."""
    avg_basket = float(np.mean([len(t) for t in transactions]))
    print("=== Synthetic Singapore Retail Transactions ===")
    print(f"  Transactions: {len(transactions):,}")
    print(f"  Products:     {len(PRODUCTS)}")
    print(f"  Avg basket:   {avg_basket:.1f} items")

    freq = product_frequency(transactions)
    n = len(transactions)
    print("\n  Top 10 products by frequency:")
    for item, count in sorted(freq.items(), key=lambda kv: -kv[1])[:10]:
        print(f"    {item:<20} {count:>5} ({count / n:.1%})")


# ════════════════════════════════════════════════════════════════════════
# RULE HELPERS
# ════════════════════════════════════════════════════════════════════════


def format_itemset(items: Iterable[str]) -> str:
    """Deterministic pretty-print of a frozenset of items."""
    return ", ".join(sorted(items))


def categorise_rule(
    antecedent: frozenset[str],
    consequent: frozenset[str],
) -> tuple[set[str], set[str], str]:
    """Return (antecedent_categories, consequent_categories, relation_type)."""
    ant_cats = {CATEGORY_MAP.get(item, "other") for item in antecedent}
    con_cats = {CATEGORY_MAP.get(item, "other") for item in consequent}
    if ant_cats == con_cats:
        rel = "within-category complement"
    elif ant_cats & con_cats:
        rel = "cross-category with overlap"
    else:
        rel = "cross-category association"
    return ant_cats, con_cats, rel


def rules_to_polars(rules: list[dict]) -> pl.DataFrame:
    """Convert a list of rule dicts into a polars DataFrame for plotting."""
    return pl.DataFrame(
        {
            "antecedent": [format_itemset(r["antecedent"]) for r in rules],
            "consequent": [format_itemset(r["consequent"]) for r in rules],
            "support": [float(r["support"]) for r in rules],
            "confidence": [float(r["confidence"]) for r in rules],
            "lift": [float(r["lift"]) for r in rules],
        }
    )


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every association-rules technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_5 lesson logs its mining + rule-evaluation + classifier runs
# into a single SQLite store so students can compare techniques side by side
# (Apriori vs FP-Growth runtime, rule-quality at varying thresholds,
# baseline vs rule-enhanced classifier metrics) after the lesson group ends.
# Mirrors m4_clustering_zoo / m4_dimreduction_zoo / m4_anomaly_zoo from
# ex_1 / ex_3 / ex_4.
#
# IMPORTANT: this store is SEPARATE from the other M4 stores. Per the
# SQLite contention trap (.session-notes), running two ExperimentTracker
# writers against the same SQLite file concurrently triggers `disk I/O
# error`. Each exercise group gets its own DB.

ASSOC_DB = "sqlite:///mlfp04_ex5_assoc_rules.db"
ASSOC_EXPERIMENT_NAME = "m4_assoc_rules_zoo"

_TRACKER_KEY_RE = re.compile(r"[^a-zA-Z0-9_.\-]")


def _slug(text: str) -> str:
    """Coerce free-form prose into a tracker-key-safe slug.

    ExperimentTracker enforces ``^[a-zA-Z_][a-zA-Z0-9_.\\-]*$`` on metric
    names. Replace forbidden characters with underscores; ensure a leading
    alpha character; collapse repeats.
    """
    s = _TRACKER_KEY_RE.sub("_", text.strip())
    s = re.sub(r"_+", "_", s).strip("_") or "metric"
    if not s[0].isalpha() and s[0] != "_":
        s = f"m_{s}"
    return s


def _finite(x: float) -> float:
    """Tracker rejects NaN/inf via MetricValueError; coerce to 0.0.

    Association-rule metrics (lift, conviction) can collapse to inf when
    confidence saturates at 1.0; aggregate scalars over empty filter sets
    can NaN. Every emit through this guard.
    """
    return float(x) if x == x and x not in (float("inf"), float("-inf")) else 0.0


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the association-rules ExperimentTracker."""
    tracker = await ExperimentTracker.create(store_url=ASSOC_DB)
    return tracker, ASSOC_EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block (or use it via ``try: … finally: teardown_engines(tracker)``).
    See ``rules/patterns.md`` § "Async Resource Cleanup": real cleanup is
    the caller's responsibility; ``close()`` must NOT live in ``__del__``.
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(_slug(name), _finite(value))
        if series_metrics:
            for name, values in series_metrics.items():
                slug = _slug(name)
                for step, value in enumerate(values, start=1):
                    await run.log_metric(slug, _finite(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 5.2: FP-Growth via mlxtend
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Use mlxtend's FP-Growth implementation end-to-end
#   - Convert polars transactions into the one-hot format FP-Growth expects
#   - Compare Apriori and FP-Growth on the same min_support
#   - Verify that both algorithms produce the same frequent itemsets
#   - Understand why FP-Growth is preferred on large transaction logs
#
# PREREQUISITES:
#   - 01_apriori_from_scratch.py (understand what frequent itemset mining does)
#
# ESTIMATED TIME: ~25 min
#
# TASKS:
#   1. Theory — FP-tree construction and recursive mining
#   2. Build — wrap mlxtend FP-Growth in a polars-friendly call
#   3. Train — run FP-Growth on 2,500 SG retail baskets
#   4. Visualise — compare Apriori vs FP-Growth itemset counts + overlap
#   5. Apply — food-delivery order-bundle mining at city scale
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import polars as pl


# ── Kailash-ML ExperimentTracker — every association-rules run logs here ─
tracker, exp_name = setup_engines()

# mlxtend's FP-Growth only accepts a pandas DataFrame and no polars
# equivalent exists, so we convert with polars' .to_pandas() at the
# mlxtend call site only. We never import pandas ourselves and stay
# polars-native everywhere else.
from mlxtend.frequent_patterns import association_rules as mlx_association_rules
from mlxtend.frequent_patterns import fpgrowth as mlx_fpgrowth

# Apriori is re-declared below (small, in-file) so this script stands
# alone without cross-file imports from a sibling starting with a digit.



## THEORY — FP-Tree and Recursive Mining


In [ ]:
# Apriori is level-wise: it scans the full database once per itemset size.
# For a transaction log with tens of millions of rows, that is the cost
# bottleneck. FP-Growth sidesteps it completely.
#
#   FP-TREE CONSTRUCTION (two passes over the data)
#   Pass 1: count single-item support; drop infrequent items.
#   Pass 2: for each transaction, sort its items by global frequency (most
#           frequent first) and insert them into a prefix tree where each
#           path represents a transaction. Shared prefixes compress.
#
#   RECURSIVE MINING (no more DB scans)
#   For each frequent item (least-frequent first), build a "conditional
#   pattern base" from the tree, then recursively mine that smaller tree.
#   No candidate generation. No extra DB passes.
#
# GUARANTEES:
#   - Same frequent itemsets as Apriori given the same min_support
#   - Typically 2-10x faster on dense data, 10-100x faster on large data
#   - Higher memory cost (the FP-tree has to fit in RAM)
#
# We use the mlxtend implementation so we can focus on the USE of the
# algorithm rather than re-implement the tree — Apriori gave us the
# mechanics in 5.1, FP-Growth gives us the speed story here.



## TASK 2 — BUILD: polars-friendly FP-Growth wrapper


Returns ``(frequent_itemsets_df, rules_df)`` as polars DataFrames.
    The mlxtend call requires pandas internally; we convert at the
    boundary so the rest of the exercise stays polars-native.


In [ ]:
def run_fp_growth(
    transactions: list[set[str]],
    min_support: float,
    min_confidence: float = 0.3,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    # TODO: build the one-hot transaction matrix with
    # transactions_to_onehot(), then convert it for mlxtend.
    # Hint: polars DataFrames have a .to_pandas() method.
    onehot_pl = ____
    onehot_pd = ____

    # TODO: call mlx_fpgrowth() with use_colnames=True so the returned
    # frame uses product names (not column indices).
    fp_frequent = ____
    # TODO: derive association rules from `fp_frequent`, filtering on
    # confidence with min_threshold=min_confidence.
    # Hint: mlx_association_rules(fp_frequent, metric="confidence", ...)
    fp_rules = ____

    frequent_pl = pl.DataFrame(
        {
            "itemset": [
                format_itemset(items) for items in fp_frequent["itemsets"].tolist()
            ],
            "size": [len(items) for items in fp_frequent["itemsets"].tolist()],
            "support": fp_frequent["support"].astype(float).tolist(),
        }
    )

    rules_pl = pl.DataFrame(
        {
            "antecedent": [
                format_itemset(items) for items in fp_rules["antecedents"].tolist()
            ],
            "consequent": [
                format_itemset(items) for items in fp_rules["consequents"].tolist()
            ],
            "support": fp_rules["support"].astype(float).tolist(),
            "confidence": fp_rules["confidence"].astype(float).tolist(),
            "lift": fp_rules["lift"].astype(float).tolist(),
        }
    )
    return frequent_pl, rules_pl



## TASK 3 — TRAIN: run FP-Growth on the same baskets used in 5.1


In [ ]:
transactions = generate_transactions(n=2500, seed=42)
print_transaction_summary(transactions)

MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.3

print("\n=== FP-Growth (mlxtend) ===")
fp_frequent_df, fp_rules_df = run_fp_growth(
    transactions, min_support=MIN_SUPPORT, min_confidence=MIN_CONFIDENCE
)
print(f"  Frequent itemsets: {fp_frequent_df.height}")
print(f"  Association rules: {fp_rules_df.height}")



## TASK 4 — VISUALISE: compare Apriori and FP-Growth on the same data


Minimal Apriori — same contract as 01_apriori_from_scratch.apriori().


In [ ]:
# We re-run Apriori here so that the comparison is self-contained. In a
# real codebase you would import the function from 01_apriori_from_scratch.

from collections import defaultdict


def _apriori(
    transactions: list[set[str]], min_support: float
) -> dict[frozenset[str], float]:
    n = len(transactions)
    min_count = min_support * n
    item_counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            item_counts[item] += 1
    freq: dict[frozenset[str], float] = {}
    level: list[frozenset[str]] = []
    for item, count in item_counts.items():
        if count >= min_count:
            fs = frozenset([item])
            freq[fs] = count / n
            level.append(fs)
    k = 2
    while level:
        prev_set = set(level)
        candidates: set[frozenset[str]] = set()
        for i, a in enumerate(level):
            for b in level[i + 1 :]:
                u = a | b
                if len(u) == k and all((u - frozenset([it])) in prev_set for it in u):
                    candidates.add(u)
        if not candidates:
            break
        counts: dict[frozenset[str], int] = defaultdict(int)
        for txn in transactions:
            tf = frozenset(txn)
            for c in candidates:
                if c.issubset(tf):
                    counts[c] += 1
        level = []
        for c, ct in counts.items():
            if ct >= min_count:
                freq[c] = ct / n
                level.append(c)
        k += 1
    return freq


apriori_itemsets = _apriori(transactions, min_support=MIN_SUPPORT)
apriori_set = {format_itemset(s) for s in apriori_itemsets}
fp_set = set(fp_frequent_df["itemset"].to_list())

print("\n=== Apriori vs FP-Growth ===")
print(f"  Apriori   itemsets: {len(apriori_set)}")
print(f"  FP-Growth itemsets: {len(fp_set)}")
print(f"  Intersection:       {len(apriori_set & fp_set)}")
print(f"  Apriori only:       {len(apriori_set - fp_set)}")
print(f"  FP-Growth only:     {len(fp_set - apriori_set)}")

# TODO: Jaccard agreement between the two itemset sets, guarding against
# divide-by-zero on an empty union. Hint: |A ∩ B| / max(|A ∪ B|, 1)
agreement = ____
print(f"  Jaccard agreement:  {agreement:.2%}")



### Checkpoint


In [ ]:
assert fp_frequent_df.height > 0, "FP-Growth should find at least one itemset"
assert fp_rules_df.height > 0, "FP-Growth should generate at least one rule"
assert agreement >= 0.90, (
    f"Apriori and FP-Growth should agree on >=90% of itemsets "
    f"at min_support={MIN_SUPPORT}; got {agreement:.2%}"
)
print("\n[ok] Checkpoint passed — FP-Growth matches Apriori on frequent itemsets\n")

fp_frequent_df.write_csv(OUTPUT_DIR / "fp_growth_itemsets.csv")
fp_rules_df.write_csv(OUTPUT_DIR / "fp_growth_rules.csv")
print(f"  Saved: {OUTPUT_DIR / 'fp_growth_itemsets.csv'}")
print(f"  Saved: {OUTPUT_DIR / 'fp_growth_rules.csv'}")



### Visualisation


In [ ]:
import time

import plotly.graph_objects as go

# (A) Itemset frequency bar chart — top 15 by support
top_itemsets = fp_frequent_df.sort("support", descending=True).head(15)
fig_freq = go.Figure(
    go.Bar(
        x=top_itemsets["support"].to_list(),
        y=top_itemsets["itemset"].to_list(),
        orientation="h",
        marker_color="#636EFA",
        text=[f"{s:.1%}" for s in top_itemsets["support"].to_list()],
        textposition="outside",
    )
)
fig_freq.update_layout(
    title="Top 15 Frequent Itemsets by Support (FP-Growth)",
    xaxis_title="Support",
    yaxis_title="Itemset",
    yaxis=dict(autorange="reversed"),
    margin=dict(l=200),
)
freq_path = OUTPUT_DIR / "02_itemset_frequency.html"
fig_freq.write_html(str(freq_path))
print(f"[viz] Itemset frequency: {freq_path}")

# (B) Apriori vs FP-Growth speed comparison across data sizes
sizes = [500, 1000, 1500, 2000, 2500]
apriori_times: list[float] = []
fp_times: list[float] = []
for sz in sizes:
    txns = generate_transactions(n=sz, seed=42)
    oh_pl = transactions_to_onehot(txns)
    oh_pd = oh_pl.to_pandas()

    t0 = time.perf_counter()
    _apriori(txns, min_support=MIN_SUPPORT)
    apriori_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    mlx_fpgrowth(oh_pd, min_support=MIN_SUPPORT, use_colnames=True)
    fp_times.append(time.perf_counter() - t0)

fig_speed = go.Figure()
fig_speed.add_trace(
    go.Scatter(
        x=sizes,
        y=apriori_times,
        mode="lines+markers",
        name="Apriori (from scratch)",
        marker_color="#EF553B",
    )
)
fig_speed.add_trace(
    go.Scatter(
        x=sizes,
        y=fp_times,
        mode="lines+markers",
        name="FP-Growth (mlxtend)",
        marker_color="#00CC96",
    )
)
fig_speed.update_layout(
    title="Apriori vs FP-Growth: Runtime by Transaction Count",
    xaxis_title="Number of Transactions",
    yaxis_title="Time (seconds)",
)
speed_path = OUTPUT_DIR / "02_speed_comparison.html"
fig_speed.write_html(str(speed_path))
print(f"[viz] Speed comparison: {speed_path}")

# INTERPRETATION: Both algorithms are level-complete — they find ALL
# frequent itemsets at the given min_support, and the sets should agree
# (modulo floating-point edge cases right at the threshold). The choice
# between them is an engineering decision about speed and memory, not
# correctness. On this small 2,500-row basket, Apriori is fine; at 10M+
# rows, FP-Growth wins because it reads the data only twice (two-pass
# tree construction) instead of once per itemset level.



## TASK 5 — APPLY: food-delivery order-bundle mining at city scale


In [ ]:
# SCENARIO: Consider a Singapore food-delivery platform handling, for
# illustration, ~800,000 orders per day across 12,000+ merchants (assumed
# figures). The merchant-growth team wants to find high-lift
# menu-item pairings (e.g., "chicken rice + iced milo") so merchants can
# publish bundle deals that raise average order value.
#
# Why FP-Growth is the right tool here:
#   - 800K orders/day * 7 days = ~5.6M transactions per weekly mining job
#   - Typical basket has 2-4 items, item universe ~100K SKUs across all
#     merchants — too large for level-wise Apriori to scan repeatedly
#   - Batch job runs overnight on a single worker; minutes matter
#   - FP-Growth builds the tree in memory in two passes, then mines it
#     without further DB scans — on large, sparse data like this it is
#     typically many times faster than Apriori (measure it: see the
#     speed sweep above)
#
# BUSINESS IMPACT (illustrative assumptions, not measured figures): if
# bundles adopted by 10% of merchants lifted those merchants' average
# order value by 5%, then at an assumed S$18 average order and 800K
# orders/day the platform would gain ~S$72K/day in GMV (800K x 10% x
# S$18 x 5%), roughly S$2.2M/month, for a mining job that costs a few
# dollars of compute per run. Confirm the AOV lift with a merchant-level
# A/B test before rolling out.
#
# LIMITATIONS:
#   - FP-tree must fit in memory; for datasets with 100M+ rows and dense
#     baskets, partitioning by merchant category is required
#   - Frequent itemsets alone do not tell you which bundles are
#     profitable — you still need to multiply by margin and demand
#     elasticity (out of scope for this exercise)



## TRACK — Log this lesson's run to the kailash-ml ExperimentTracker


In [ ]:
# Logs FP-Growth's headline counts + the per-size runtime sweep against
# Apriori. Series = runtimes at each transaction-count tested; scalars =
# itemset / rule counts + Jaccard agreement vs the hand-rolled Apriori.

# TODO: speedup at the largest size = Apriori time / FP-Growth time
# (return 0.0 if the FP-Growth time is zero).
speedup = ____
track_run(
    tracker,
    exp_name,
    run_name=____,  # TODO: a stable name, e.g. "fp_growth_mlxtend"
    params={
        "algorithm": "fp_growth",
        "implementation": "mlxtend",
        "n_transactions": len(transactions),
        "min_support": MIN_SUPPORT,
        "min_confidence": MIN_CONFIDENCE,
        "swept_sizes": ",".join(str(s) for s in sizes),
    },
    scalar_metrics={
        "n_frequent_itemsets": float(fp_frequent_df.height),
        "n_rules": float(fp_rules_df.height),
        "jaccard_apriori_vs_fp": float(agreement),
        "apriori_only_count": float(len(apriori_set - fp_set)),
        "fp_only_count": float(len(fp_set - apriori_set)),
        "speedup_at_max_size": float(speedup),
        "apriori_runtime_at_max": float(apriori_times[-1]),
        "fp_runtime_at_max": float(fp_times[-1]),
    },
    series_metrics={
        "apriori_runtime_seconds": [float(t) for t in apriori_times],
        "fp_growth_runtime_seconds": [float(t) for t in fp_times],
    },
)
print(f"  [tracked] FP-Growth + speed sweep logged to {exp_name}\n")



## DESTINATION-FIRST CLOSE — the mlxtend.frequent_patterns one-liner


In [ ]:
# This lesson built a polars wrapper around mlxtend FP-Growth, ran the
# Apriori-vs-FP-Growth speed sweep, and verified itemset agreement —
# ~270 lines of structure to internalise WHY FP-Growth is the production
# choice on dense, city-scale data. The destination is exactly the two
# library calls you saw inside ``run_fp_growth``:
#
#   from mlxtend.frequent_patterns import fpgrowth, association_rules
#   itemsets = fpgrowth(onehot_df, min_support=0.03, use_colnames=True)
#   rules    = association_rules(itemsets, metric='confidence', min_threshold=0.3)
#
# Two calls, one pandas frame in, two pandas frames out. The polars
# wrapper exists for ergonomics, not for FP-Growth's correctness — every
# downstream lesson (03 evaluation, 04 features) consumes that same rule
# table.

print("  Destination contract:")
print("    fpgrowth(onehot_df, min_support, use_colnames=True)  -> itemsets")
print("    association_rules(itemsets, metric, min_threshold)   -> rules")
print(
    f"  Today's run: {fp_frequent_df.height} itemsets, "
    f"{fp_rules_df.height} rules — both shape: pandas DataFrame"
)
print()



## REFLECTION


[x] Wrapped mlxtend FP-Growth in a polars-friendly call boundary
  [x] Converted basket-sets into the one-hot format FP-Growth expects
  [x] Verified that FP-Growth and Apriori agree on frequent itemsets
  [x] Identified a city-scale workload (food-delivery bundles) where
      FP-Growth's two-pass tree construction is the economic
      difference-maker
  [x] Locked in the production destination: two mlxtend calls
      (fpgrowth + association_rules) feed every downstream lesson

  KEY INSIGHT: Both algorithms are correct. The choice between them is
  about the SHAPE of your data (size, density, access cost) and never
  about which one "finds more patterns." When someone tells you
  FP-Growth "discovered" rules Apriori missed, the real difference is
  almost always a different min_support.

  Next: 03_rule_evaluation.py — turn frequent itemsets into association
  rules and score them with support, confidence, lift, and conviction.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

